In [10]:
from pdfminer.high_level import extract_text
import ebooklib
from ebooklib import epub
from bs4 import BeautifulSoup
import re

#### PDF OCR

In [ ]:
text = extract_text("data/Draft 12_15_22 - Project Hail Mary Screenplay.pdf")
with open("phm_script_raw_draft.txt", "w", encoding="utf-8") as f:
    f.write(text)

#### EPUB to txt

In [12]:
BLOCK_TAGS = {"p", "div", "br", "h1", "h2", "h3", "h4", "h5", "h6", "li", "tr"}

def epub_to_text(epub_path):
    book = epub.read_epub(epub_path)
    chapters = []

    for item in book.get_items():
        if item.get_type() == ebooklib.ITEM_DOCUMENT:
            soup = BeautifulSoup(item.get_content(), "html.parser")

            # Insert newline markers at block boundaries only
            for tag in soup.find_all(BLOCK_TAGS):
                tag.insert_before("\n")
                tag.insert_after("\n")

            # Now get_text with no separator — inline tags just yield their text naturally
            text = soup.get_text(separator="")

            # Clean up
            text = re.sub(r" {2,}", " ", text)        # collapse extra spaces
            text = re.sub(r"\n{3,}", "\n\n", text)    # collapse extra blank lines
            text = re.sub(r" \n", "\n", text)         # trailing spaces before newlines
            text = re.sub(r"\n ", "\n", text)         # leading spaces after newlines

            chapters.append(text.strip())

    return "\n\n".join(chapters)

full_text = epub_to_text("data/Project Hail Mary.epub")
with open("book_raw.txt", "w", encoding="utf-8") as f:
    f.write(full_text)